# Phase 11: Final Error Analysis and Model Interpretability
## WaterGuardX — Autonomous Water Distribution System Anomaly Detection

### Section 1: Objective
The primary objective of Phase 11 is to conduct a rigorous, evidence-based diagnostic analysis of the finalized anomaly-detection model. Specifically, this phase aims to investigate:
1. **Error Topology**: Where, when, and why the model makes classification mistakes (False Positives and False Negatives).
2. **Sensor Vulnerabilities**: Which sensor features and physical parameters exhibit anomalous divergence during error events.
3. **Distribution Shift Impact**: How physical sensor drift changes the error distribution, and how adaptation mitigates false alarms.
4. **Model Interpretability**: Unpacking the model's internal representations through:
   - Self-attention weight visualization across Transformer encoder layers.
   - Temporal occlusion sensitivity (measuring the influence of temporal sub-windows).
   - Sensor permutation sensitivity (quantifying input feature impact on anomaly scores).
5. **Confidence & Calibration**: Analyzing predicted probability calibration via Expected Calibration Error (ECE) and reliability diagrams.

**Core Academic Principle**: All findings are empirical. Attention maps indicate model sensitivity and focus regions, but are **not** claimed as causal explanations of physical hydraulic phenomena.


### Section 2: Final Model Architecture & Checkpoints

From Phase 10, the selected optimal model is the **Adaptive SSL Temporal Transformer**:
- **Backbone**: 3-layer Transformer Encoder with Sinusoidal Positional Encoding.
- **Parameters**: 19,313 active parameters.
- **Dimensionality**: $d_{\text{model}} = 32$, $h = 2$ heads, $d_{\text{ff}} = 64$, dropout $= 0.15$.
- **Sequence Length**: 60 timesteps (5 hours at 5-minute sampling interval).
- **Checkpoints**: `models/final_model.pt` (Nominal), `models/final_model_adapted.pt` (Adapted).
- **Calibrated Thresholds**: $\tau_{\text{nominal}} = 0.0100$, $\tau_{\text{adapted}} = 0.0100$.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from IPython.display import Image, display

# Verify data availability
err_res_dir = Path('../results/error_analysis')
err_fig_dir = Path('../figures/error_analysis')

preds_path = err_res_dir / 'final_predictions.csv'
fp_path = err_res_dir / 'false_positives.csv'
fn_path = err_res_dir / 'false_negatives.csv'
err_sum_path = err_res_dir / 'error_summary.csv'
interp_path = err_res_dir / 'interpretability_results.csv'
sensor_div_path = err_res_dir / 'sensor_error_divergence.csv'

print(f"Error Analysis Results Directory: {err_res_dir.resolve()}")
print(f"All artifacts present: {all(p.exists() for p in [preds_path, fp_path, fn_path, err_sum_path, interp_path, sensor_div_path])}")


### Section 3: Final Test Predictions Generation

Predictions were generated on the untouched held-out December 2018 streams across three operating conditions:
1. `Normal_Test_Nominal`: Untouched nominal December stream (8,881 sequences) evaluated with the nominal model.
2. `Shifted_Test_PreAdapt`: Untouched shifted December stream evaluated with the static nominal model (pre-adaptation).
3. `Shifted_Test_PostAdapt`: Untouched shifted December stream evaluated with the adapted model (post-adaptation).


In [ ]:
preds_df = pd.read_csv(preds_path)
print(f"Total Prediction Records: {len(preds_df):,}")
preds_df.groupby('eval_condition')['error_type'].value_counts().unstack().fillna(0).astype(int)


### Section 4: Confusion-Matrix Error Analysis

On the adapted shifted evaluation stream (8,881 total sequences, 1,681 true anomalies, 7,200 true normal sequences):
- **True Positives (TP)**: 1,652 sequences (98.27% sensitivity / recall).
- **True Negatives (TN)**: 7,137 sequences (99.12% specificity).
- **False Positives (FP)**: 63 sequences (0.88% False Positive Rate).
- **False Negatives (FN)**: 29 sequences (1.73% False Negative Rate).


In [ ]:
display(Image(filename='../figures/error_analysis/confusion_matrix_detailed.png'))


### Section 5: In-Depth False-Positive Analysis

Representative False Positives (normal operating sequences classified as anomalies):
1. **Hydraulic Switching Transients**: Rapid valve throttling or pump staging produces sharp pressure gradients ($> 2.0\sigma$) that momentarily mimic leak/burst profiles.
2. **Borderline Threshold Fluctuations**: Predictions where the anomaly score ($0.0100 - 0.0200$) is marginally above the calibrated threshold.
3. **Residual Sensor Drift**: Localized sensor variance in pressure nodes `n215` and `p235`.


In [ ]:
fp_df = pd.read_csv(fp_path)
fp_df[['Error ID', 'Timestamp', 'Probability', 'Observed Pattern']].head(8)


In [ ]:
display(Image(filename='../figures/error_analysis/representative_false_positive.png'))


### Section 6: In-Depth False-Negative Analysis

Representative False Negatives (anomalous sequences missed by the model):
1. **Subtle / Incipient Leakage**: Leakage events with low hydraulic deviation (mean anomaly magnitude $< 0.60\sigma$), which remain within normal diurnal demand noise bounds.
2. **Borderline False Negatives**: Sequences where the anomaly probability ($0.0050 - 0.0099$) sits just below the $0.0100$ threshold.


In [ ]:
fn_df = pd.read_csv(fn_path)
fn_df[['Error ID', 'Timestamp', 'Probability', 'Observed Pattern']].head(8)


In [ ]:
display(Image(filename='../figures/error_analysis/representative_false_negative.png'))


### Section 7: Error Distribution Under Distribution Shift

Contrasting error distributions before and after adaptation under severe sensor distribution shift ($k = 1.0\sigma$ drift in pH, turbidity, and conductivity):
- **Pre-Adaptation**: The static model produces **7,200 False Positives (100% FPR)**, rendering the system unusable.
- **Post-Adaptation**: Controlled fine-tuning with 20% replay buffer suppresses False Positives down to **63 (0.88% FPR)**, representing a **99.1% reduction in false alarms** while maintaining $98.27\%$ recall.


In [ ]:
display(Image(filename='../figures/error_analysis/error_distribution_normal_vs_shifted.png'))


### Section 8: Temporal Error Analysis & Operating Cycles

Errors exhibit distinct temporal concentration patterns:
- False Positives cluster predominantly during early morning (05:00 - 07:00) and evening (18:00 - 20:00) peak diurnal demand transitions.
- Missed anomalies (FN) occur primarily during nighttime baseline periods (01:00 - 04:00) when background flow rate is minimal and small pressure drops are attenuated.


In [ ]:
# Analyze timestamp distribution of errors
preds_adapted = preds_df[preds_df['eval_condition'] == 'Shifted_Test_PostAdapt'].copy()
preds_adapted['hour'] = pd.to_datetime(preds_adapted['timestamp']).dt.hour

hourly_errors = preds_adapted.groupby('hour')['error_type'].value_counts().unstack().fillna(0).astype(int)
hourly_errors[['False_Positive', 'False_Negative']].plot(kind='bar', figsize=(11, 4), color=['#e74c3c', '#f39c12'])
plt.title("Hourly Distribution of Errors (Adapted Shifted Stream - December 2018)")
plt.xlabel("Hour of Day (UTC)")
plt.ylabel("Error Count")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### Section 9: Sensor-Level Error Divergence Analysis

Comparing the sensor value distributions across True Negatives (TN), False Positives (FP), True Positives (TP), and False Negatives (FN):
- The largest divergence during False Positives occurs in pressure sensors `n215`, `p235`, and `n1`.
- False Negatives show sensor profiles nearly indistinguishable from normal operating baselines.


In [ ]:
sensor_div = pd.read_csv(sensor_div_path)
sensor_div[['sensor', 'mean_TN', 'mean_FP', 'fp_divergence', 'mean_TP', 'mean_FN', 'fn_divergence']].head(10)


### Section 10: Model Interpretability Methodology

We implement three complementary interpretability techniques tailored to the Transformer architecture:
1. **Sensor Permutation Sensitivity**: Quantifying the drop in anomaly probability and PR-AUC when each sensor channel is shuffled across evaluation sequences.
2. **Temporal Occlusion**: Sliding a 5-timestep temporal mask across the 60-timestep sequence to observe the localized impact on anomaly probability.
3. **Multi-Head Self-Attention Extraction**: Extracting the $60 \times 60$ self-attention matrix from each Transformer encoder layer to visualize temporal attention distribution.


### Section 11: Interpretability Results & Findings

- **Sensor Importance**: The most influential sensor features are `p227` (flow), `n1` (pressure), and `p235` (pressure), which exhibit the highest probability sensitivity when perturbed.
- **Temporal Importance**: Occlusion sensitivity reveals that the most recent 15 timesteps (last 75 minutes) exert the strongest influence on the classification output.
- **Attention Visualization**: Attention weights in Layer 1 focus on local temporal continuity, whereas Layer 2 captures cross-timestep periodic dependencies.


In [ ]:
interp_df = pd.read_csv(interp_path)
interp_df[['rank', 'feature', 'mean_abs_prob_delta', 'mean_signed_delta', 'ap_drop']].head(10)


In [ ]:
display(Image(filename='../figures/error_analysis/sensor_sensitivity_importance.png'))
display(Image(filename='../figures/error_analysis/temporal_occlusion_importance.png'))
display(Image(filename='../figures/error_analysis/attention_map_representative.png'))


### Section 12: Confidence & Calibration Analysis

- **Bimodal Probability Distribution**: The model generates sharp bimodal probability predictions (concentrated near 0.0 for normal sequences and near 1.0 for anomaly sequences).
- **Expected Calibration Error (ECE)**:
  - Normal Stream ECE: $\approx 0.012$
  - Adapted Stream ECE: $\approx 0.018$
- Errors are predominantly concentrated in the low-confidence boundary regime ($0.005 \le P \le 0.05$).


In [ ]:
display(Image(filename='../figures/error_analysis/prediction_probability_distribution.png'))
display(Image(filename='../figures/error_analysis/calibration_reliability_diagram.png'))


### Section 13: Key Findings & Error Categorization

Summary of evidence-based error categorization across all false alarms and missed detections:


In [ ]:
err_summary = pd.read_csv(err_sum_path)
err_summary[['Error Type', 'Category', 'Count', 'Percentage of Type', 'Percentage of Stream']]


### Section 14: Technical Limitations & Non-Causality Statement

1. **Non-Causality**: Permutation sensitivity and self-attention weights measure statistical model dependencies; they **do not** prove physical causality in the water distribution network.
2. **Incipient Anomaly Detection Boundary**: Anomalies with pressure drops $< 0.3\sigma$ remain near the physical observability limit of SCADA telemetry at 5-minute sampling resolution.
3. **Phase 11 Complete**: All error logs, representative sequences, divergence metrics, and interpretability figures are saved under `results/error_analysis/` and `figures/error_analysis/`.
